# Notebook 3 — GNN Detection Models (GCN / GIN / SAT-Graph)
### Reconstruction-based anomaly detection on the ontology graph

This is the **headline** task. Each model is a **graph autoencoder**: it encodes every window's 30
node-features through message passing on the ontology graph, squeezes them to a small latent, and
reconstructs them. Trained **only on normal windows**, the model learns the normal inter-variable
structure; when an event breaks that structure (July-9 PM artifact, the fire, the outage), the
reconstruction error spikes — that spike is the detection signal.

- **Input / output**: the 8 node-feature channels per variable (`mean, std, …, state_sev`).
- **Loss**: `MSELoss` on normal windows only.
- **Anomaly score**: per-window mean reconstruction error.
- **Three encoders**: `GCN`, `GIN`, and `SAT-Graph` (a Structure-Aware Transformer — `TransformerConv`
  with degree + random-walk positional encodings, the architecture from the earlier NB-03).

**Per-deployment standardisation.** The lab and apartment have different baselines (lab PM ≈ 0,
apartment PM ≈ 16). So we re-standardise each deployment's features using **its own** normal-train
statistics (from `X_raw`), then pool the now-comparable windows to train one model per architecture.
This avoids the apartment's normal looking anomalous to a lab-scaled model.


In [ ]:
import json, time, copy, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GINConv, TransformerConv, BatchNorm, LayerNorm

warnings.filterwarnings("ignore")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

## 1. Load the graph substrate (Notebook 2)

In [ ]:
_C = [Path("processed/graph"), Path("processed"), Path(".")]
GP = next((p for p in _C if (p / "graph_windows.npz").exists()), Path("processed/graph"))
OUT = GP
d = np.load(GP / "graph_windows.npz", allow_pickle=True)
man = json.load(open(GP / "graph_manifest.json"))

X_raw = d["X_raw"]                         # (W, N, F) unstandardised
edge_index_np = d["edge_index"]
event_id = d["event_id"]; anomaly = d["anomaly"]
location = d["location"].astype(str)
det_train = d["det_train"]; det_val = d["det_val"]; det_test = d["det_test"]
W, N_NODES, F_CH = X_raw.shape
FEATURES = man["feature_channels"]; EVENT_ID = man["event_id_map"]
print(f"Windows={W}  Nodes={N_NODES}  Channels={F_CH} {FEATURES}")
print(f"Edges={edge_index_np.shape[1]} | det split train/val/test = "
      f"{det_train.sum()}/{det_val.sum()}/{det_test.sum()} (test events={int((event_id[det_test]!=0).sum())})")

## 2. Per-deployment standardisation (train-only stats)
Channels 0–6 are standardised per `(deployment, node, channel)` using that deployment's
**detection-train (normal)** windows; `state_sev` (channel 7) is already 0–1 and left as is.


In [ ]:
X = X_raw.copy()
for loc in ["laboratory", "apartment"]:
    locmask = location == loc
    trmask = locmask & det_train
    if trmask.sum() == 0:
        continue
    mu = X_raw[trmask][:, :, :7].mean(axis=0)                 # (N,7)
    sd = X_raw[trmask][:, :, :7].std(axis=0); sd[sd < 1e-8] = 1.0
    X[locmask, :, :7] = (X_raw[locmask][:, :, :7] - mu) / sd
X = X.astype(np.float32)
print("Standardised per deployment. Global train channel-mean ≈",
      round(float(X[det_train][:, :, :7].mean()), 4))

## 3. Build PyG graphs (one per window, shared topology)

In [ ]:
edge_index = torch.tensor(edge_index_np, dtype=torch.long)
data_all = [Data(x=torch.tensor(X[w], dtype=torch.float), edge_index=edge_index) for w in range(W)]

def loader_for(mask, shuffle):
    subset = [data_all[i] for i in np.where(mask)[0]]
    return DataLoader(subset, batch_size=128, shuffle=shuffle)

train_loader = loader_for(det_train, True)
val_loader   = loader_for(det_val, False)
all_loader   = DataLoader(data_all, batch_size=256, shuffle=False)   # for per-window errors

# Positional encodings for SAT (computed once on the 30-node ontology graph)
def degree_pe(ei, n):
    deg = torch.zeros(n); deg.scatter_add_(0, ei[0], torch.ones(ei.size(1)))
    return (deg / max(deg.max().item(), 1.0)).unsqueeze(1)            # (n,1)
def rwse_pe(ei, n, k=8):
    deg = torch.zeros(n); deg.scatter_add_(0, ei[0], torch.ones(ei.size(1)))
    val = (1.0 / (deg + 1e-8))[ei[0]]
    P = torch.sparse_coo_tensor(ei, val, (n, n)).coalesce().to_dense()
    rw = torch.zeros(n, k); Pk = P.clone()
    for i in range(k):
        rw[:, i] = Pk.diagonal(); Pk = Pk @ P
    return rw                                                          # (n,k)
PE = torch.cat([degree_pe(edge_index, N_NODES), rwse_pe(edge_index, N_NODES, 8)], dim=1)
print(f"Built {len(data_all)} window-graphs | SAT positional-encoding dim = {PE.shape[1]}")

## 4. The graph autoencoder (GCN / GIN / SAT-Graph encoders)
Encoder = message passing on the ontology graph → latent bottleneck. Decoder = node-wise MLP back to
the 8 channels. A node's reconstruction therefore depends on its ontology neighbours, so a broken
relationship raises the error.


In [ ]:
class GraphAE(nn.Module):
    def __init__(self, in_ch, hid=64, latent=16, conv="gcn", nl=2, heads=4, dr=0.2, pe=None):
        super().__init__()
        self.conv_type = conv; self.dr = dr
        extra = pe.shape[1] if (conv == "sat" and pe is not None) else 0
        if conv == "sat":
            self.register_buffer("pe", pe)
        self.enc_in = nn.Linear(in_ch + extra, hid)
        self.convs = nn.ModuleList(); self.norms = nn.ModuleList()
        for _ in range(nl):
            if conv == "gcn":
                self.convs.append(GCNConv(hid, hid, add_self_loops=True)); self.norms.append(BatchNorm(hid))
            elif conv == "gin":
                mlp = nn.Sequential(nn.Linear(hid, 2 * hid), nn.BatchNorm1d(2 * hid), nn.ReLU(), nn.Linear(2 * hid, hid))
                self.convs.append(GINConv(mlp, train_eps=True)); self.norms.append(BatchNorm(hid))
            else:  # sat
                self.convs.append(TransformerConv(hid, hid // heads, heads=heads, dropout=dr, concat=True))
                self.norms.append(LayerNorm(hid))
        self.to_latent = nn.Linear(hid, latent)
        self.dec = nn.Sequential(nn.Linear(latent, hid), nn.ReLU(), nn.Dropout(dr), nn.Linear(hid, in_ch))

    def forward(self, x, ei):
        h = x
        if self.conv_type == "sat":
            B = x.size(0) // self.pe.size(0)
            h = torch.cat([x, self.pe.repeat(B, 1)], dim=-1)
        h = F.relu(self.enc_in(h)); h = F.dropout(h, self.dr, self.training)
        for c, n in zip(self.convs, self.norms):
            h = F.relu(n(c(h, ei))); h = F.dropout(h, self.dr, self.training)
        return self.dec(self.to_latent(h))

# numpy ranking metrics (no sklearn dependency)
def auc_score(y, s):
    y = np.asarray(y); s = np.asarray(s); pos = s[y == 1]; neg = s[y == 0]
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    allv = np.concatenate([pos, neg]); ranks = allv.argsort().argsort() + 1
    u = ranks[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2
    return float(u / (len(pos) * len(neg)))
def ap_score(y, s):
    y = np.asarray(y); s = np.asarray(s)
    if y.sum() == 0: return float("nan")
    o = np.argsort(-s); y = y[o]; tp = np.cumsum(y); fp = np.cumsum(1 - y)
    prec = tp / (tp + fp); rec = tp / y.sum(); rprev = np.concatenate([[0], rec[:-1]])
    return float(((rec - rprev) * prec).sum())
print("GraphAE + metrics defined.")

## 5. Training loop (reconstruct normal windows)

In [ ]:
def train_ae(model, name, epochs=120, patience=15, lr=1e-3):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=8, min_lr=1e-6)
    crit = nn.MSELoss()
    best = 1e9; best_state = None; pc = 0; hist = {"train": [], "val": []}; t0 = time.time()
    print(f"\n=== {name} ===")
    for ep in range(1, epochs + 1):
        model.train(); tl = []
        for b in train_loader:
            b = b.to(DEVICE); opt.zero_grad()
            loss = crit(model(b.x, b.edge_index), b.x)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl.append(loss.item())
        model.eval(); vl = []
        with torch.no_grad():
            for b in val_loader:
                b = b.to(DEVICE); vl.append(crit(model(b.x, b.edge_index), b.x).item())
        tr, va = float(np.mean(tl)), float(np.mean(vl)); sched.step(va)
        hist["train"].append(tr); hist["val"].append(va)
        if va < best - 1e-6: best = va; best_state = copy.deepcopy(model.state_dict()); pc = 0
        else: pc += 1
        if ep % 20 == 0 or ep == 1: print(f"  ep{ep:3d}  train={tr:.4f}  val={va:.4f}")
        if pc >= patience: print(f"  early stop @ {ep}"); break
    model.load_state_dict(best_state)
    print(f"  done {time.time()-t0:.1f}s  best_val={best:.4f}")
    return model, hist

@torch.no_grad()
def recon_errors(model):
    model.eval(); out = []
    for b in all_loader:
        b = b.to(DEVICE)
        e = ((model(b.x, b.edge_index) - b.x) ** 2).mean(dim=1)       # per node
        out.append(e.view(-1, N_NODES).mean(dim=1).cpu().numpy())     # per window
    return np.concatenate(out)
print("Training utilities ready.")

## 6. Train GCN, GIN, SAT-Graph

In [ ]:
CONFIGS = {
    "GCN":       dict(conv="gcn", hid=64, latent=16, nl=2),
    "GIN":       dict(conv="gin", hid=64, latent=16, nl=2),
    "SAT-Graph": dict(conv="sat", hid=64, latent=16, nl=2, heads=4),
}
MODELS, HIST = {}, {}
for name, kw in CONFIGS.items():
    pe = PE if kw["conv"] == "sat" else None
    m = GraphAE(F_CH, pe=pe, **kw)
    MODELS[name], HIST[name] = train_ae(m, name)
    torch.save(MODELS[name].state_dict(), OUT / f"{name.lower().replace('-','_')}_ae.pt")
print("\nSaved model weights to", OUT.resolve())

## 7. Quick detection scores

The anomaly threshold is the 99th percentile of **validation (normal)** reconstruction error — i.e.
calibrated to ~1% false positives on clean data. We then report, on the detection test set: ROC-AUC
and PR-AUC (error vs. the binary anomaly label), the false-positive rate on normal test windows, and
**per-event recall** (the fraction of fire / dust / outage windows flagged). Full ROC/PR curves, the
localisation of *which variable* drove each detection, and the final taux come in Notebook 4.


In [ ]:
import pandas as pd
THRESH_Q = 0.99
ERR = {}; rows = []
for name, model in MODELS.items():
    err = recon_errors(model); ERR[name] = err
    thr = float(np.quantile(err[det_val], THRESH_Q))
    te = det_test
    row = {"Model": name,
           "ROC_AUC": round(auc_score(anomaly[te], err[te]), 4),
           "PR_AUC": round(ap_score(anomaly[te], err[te]), 4),
           "threshold": round(thr, 5),
           "FPR_normal": round(float(((err[te] > thr) & (anomaly[te] == 0)).sum() / max((anomaly[te] == 0).sum(), 1)), 4)}
    for ev, eid in {"fire": 1, "dust_humidity": 2, "power_outage": 3}.items():
        m = (event_id == eid) & te
        row[f"{ev}_recall"] = round(float((err[m] > thr).mean()), 3) if m.sum() > 0 else None
    rows.append(row)

det_df = pd.DataFrame(rows)
det_df.to_csv(OUT / "nb03_detection_quick.csv", index=False)
np.savez_compressed(OUT / "recon_errors.npz",
                    **{k: v for k, v in ERR.items()},
                    det_val=det_val, det_test=det_test, event_id=event_id, anomaly=anomaly, location=location)
print("=== Quick detection scores (test set) ===")
print(det_df.to_string(index=False))
print("\nSaved: nb03_detection_quick.csv, recon_errors.npz, *_ae.pt")

## 8. Training curves & reconstruction-error separation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
colors = {"GCN": "#378ADD", "GIN": "#1D9E75", "SAT-Graph": "#D85A30"}
for name, h in HIST.items():
    axes[0].plot(h["val"], color=colors[name], lw=1.4, label=name)
axes[0].set_title("Validation reconstruction MSE (normal windows)")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("val MSE"); axes[0].legend()

best_name = det_df.sort_values("ROC_AUC", ascending=False)["Model"].iloc[0]
err = ERR[best_name]; te = det_test
axes[1].hist(err[te][anomaly[te] == 0], bins=40, alpha=0.6, color="#378ADD", label="normal", density=True)
axes[1].hist(err[te][anomaly[te] == 1], bins=40, alpha=0.6, color="#D85A30", label="event", density=True)
axes[1].axvline(np.quantile(err[det_val], THRESH_Q), color="black", ls="--", lw=1, label="threshold")
axes[1].set_title(f"{best_name} — reconstruction error: normal vs event (test)")
axes[1].set_xlabel("per-window reconstruction error"); axes[1].legend()
fig.tight_layout(); plt.savefig(OUT / "nb03_detection_curves.png", dpi=130, bbox_inches="tight"); plt.show()
print("Saved: nb03_detection_curves.png  | best model by ROC-AUC:", best_name)

## 9. Summary & hand-off

**Produced** (in `processed/graph/`)
- `gcn_ae.pt`, `gin_ae.pt`, `sat_graph_ae.pt` — trained detection autoencoders.
- `recon_errors.npz` — per-window reconstruction error for every model + labels/splits.
- `nb03_detection_quick.csv`, `nb03_detection_curves.png` — quick scores and error-separation plot.

**What this notebook establishes** — the headline detection pipeline runs: each GNN learns normal
inter-variable structure on the ontology graph and flags windows whose structure breaks, scored by
ROC-AUC / PR-AUC and per-event recall.

**Next**
- **Notebook 4 — Detection evaluation**: ROC/PR curves, per-event detection tables, *which-variable*
  localisation (per-node reconstruction error on the July-9 / fire / outage windows), and the final
  **taux** (event detection rate).
- **Notebook 5 — Prediction comparison**: the leakage-clean pollutant regression (GNN vs RNN/LSTM/GRU)
  using the same graph, evaluated with R²/MAE/RMSE against Xuanzhe.
